# 04 — L2 Regularization + Threshold Tuning

**Goal:** improve generalization.

L2 regularization adds a penalty on large weights:

\[
\mathcal{L}(w, b) = \text{BCE}(w,b) + \frac{\lambda}{2}\|w\|_2^2
\]

This often:
- reduces overfitting
- makes the solution more stable

We will also show that the default decision threshold 0.5 is arbitrary; for some applications,
you may choose a different threshold to trade off precision vs recall.


In [ ]:
import numpy as np

from sklearn.datasets import load_breast_cancer

from utils.data import train_test_split, StandardScaler
from utils.metrics import accuracy
from scripts.logreg_scratch import LogisticRegressionScratch, FitConfig


## Data + scaling

In [ ]:
data = load_breast_cancer()
X = data.data.astype(float)
y = data.target.astype(int)

X_train_full, X_test, y_train_full, y_test = train_test_split(X, y, test_size=0.2, seed=42)

# validation split from train
X_train, X_val, y_train, y_val = train_test_split(X_train_full, y_train_full, test_size=0.2, seed=123)

scaler = StandardScaler().fit(X_train)
X_train = scaler.transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)


## Train with L2

In [ ]:
lambdas = [0.0, 1e-4, 1e-3, 1e-2]
results = []

for lam in lambdas:
    model = LogisticRegressionScratch()
    cfg = FitConfig(lr=0.1, epochs=600, batch_size=64, l2_lambda=lam, verbose=False)
    model.fit(X_train, y_train, X_val=X_val, y_val=y_val, config=cfg)

    val_pred = model.predict(X_val)
    val_acc = accuracy(y_val, val_pred)
    results.append((lam, val_acc, model))

best_lam, best_val_acc, best_model = sorted(results, key=lambda t: t[1], reverse=True)[0]
print('Best lambda:', best_lam, 'val_acc:', best_val_acc)

test_pred = best_model.predict(X_test)
print('Test accuracy:', accuracy(y_test, test_pred))


## Threshold tuning (simple demo)

We’ll try a few thresholds and pick the one that maximizes validation accuracy.
In practice, you might optimize F1 or recall instead.

In [ ]:
thresholds = np.linspace(0.1, 0.9, 17)
val_scores = []

val_proba = best_model.predict_proba(X_val)
for t in thresholds:
    val_pred_t = (val_proba >= t).astype(int)
    val_scores.append(accuracy(y_val, val_pred_t))

best_t = float(thresholds[int(np.argmax(val_scores))])
print('Best threshold:', best_t, 'val_acc:', max(val_scores))

test_pred_t = best_model.predict(X_test, threshold=best_t)
print('Test accuracy (threshold tuned):', accuracy(y_test, test_pred_t))


Next: in Notebook **05**, we’ll compute confusion matrix + precision/recall/F1 from scratch and plot learning curves.